# L01 · Tokens, probabilities, autoregressive LMs

## Goal

**Estimated time:** 30 min · **Path:** full

- turn logits into probabilities
- read next-token tensor shapes
- verify causal prefixes

### Current position: L00 → **L01** → L02

```text
Prompt/Data -> state source -> ... -> L01 -> ... -> fair evaluation
```

Alt text: The course map highlights L01 between its prerequisite and next lesson; every method remains connected to the same evaluation stage.

## Setup

In [1]:
LESSON_ID = "L01"
from pathlib import Path
import sys
import torch

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = Path.cwd().parents[1]
sys.path.insert(0, str(repo_root / "src"))

import opd_study
from opd_study.device import resolve_device
from opd_study.utils import seed_everything

seed_everything(42)
device_report = resolve_device("cpu")
print({"lesson": LESSON_ID, "opd_study": opd_study.__version__,
       "torch": torch.__version__, "device": device_report.selected,
       "profile": "toy", "network": "not required"})

{'lesson': 'L01', 'opd_study': '0.1.0.dev0', 'torch': '2.13.0', 'device': 'cpu', 'profile': 'toy', 'network': 'not required'}


## Steps

### 1/3 · 8–12 min

A language model emits one vocabulary-wide logit vector at each position. Softmax turns it into a conditional distribution summing to one; the causal mask prevents future-token leakage.

Figure alt: labels and numbers remain readable without color.

### Core mechanics

An autoregressive model factors sequence probability as `p(x_1:T)=Π_t p(x_t | x_<t)`. Logits have shape `[batch, time, vocabulary]`; logits at position `t` predict target `t+1`. Missing this one-position shift accidentally trains with future-token leakage.

The padding mask removes nonexistent positions, the response mask excludes prompts from the objective, and the causal mask prevents attention to the future. They solve different problems and cannot replace one another.

### Production implementation: why this design

The implementation flows through `token IDs → token/position embedding → causal Transformer → layer norm → vocabulary logits`. `TinyCausalLM.forward` validates rank/length and separates causal from padding masks. `generate` explicitly separates greedy (`temperature=0`) and stochastic sampling.

Production code: [`tiny_transformer.py`](../../src/opd_study/models/tiny_transformer.py), [`tokenizer.py`](../../src/opd_study/data/tokenizer.py).

In [2]:
import inspect
from opd_study.models import TinyCausalLM

objects_to_show = (TinyCausalLM.forward, TinyCausalLM.generate,)
for object_to_show in objects_to_show:
    source_lines = inspect.getsource(object_to_show).splitlines()
    print(f"\n# {object_to_show.__module__}.{object_to_show.__qualname__}")
    print("\n".join(source_lines[:80]))
    if len(source_lines) > 80:
        print(f"... {len(source_lines) - 80} more lines; open the linked source file")


# opd_study.models.tiny_transformer.TinyCausalLM.forward
    def forward(self, token_ids: Tensor, attention_mask: Tensor | None = None) -> Tensor:
        if token_ids.ndim != 2:
            raise ValueError("token_ids must have shape [batch, sequence]")
        _, sequence_length = token_ids.shape
        if sequence_length < 1 or sequence_length > self.config.max_sequence_length:
            raise ValueError(
                f"sequence length must be within [1, {self.config.max_sequence_length}], "
                f"got {sequence_length}"
            )
        if attention_mask is None:
            attention_mask = torch.ones_like(token_ids, dtype=torch.bool)
        if attention_mask.shape != token_ids.shape:
            raise ValueError("attention_mask must have the same shape as token_ids")
        positions = torch.arange(sequence_length, device=token_ids.device)
        hidden = self.token_embedding(token_ids) * math.sqrt(self.config.hidden_size)
        hidden = hidden + self.

### Alternatives and trade-offs

Production LLMs may use subword tokenizers, RoPE, tied embeddings, RMSNorm, and FlashAttention. This course chooses a character tokenizer and standard Transformer so token boundaries and masks stay visible. That is an educational choice, not a claim that Qwen has the same architecture.

### 2/3 · Run and observe

Predict before running: which invariant should you inspect first in L01's output? Write one sentence, then run.

In [3]:
logits = torch.tensor([2.0, 1.0, 0.0])
probabilities = torch.softmax(logits, dim=-1)
manual = logits.exp() / logits.exp().sum()
print("probabilities:", probabilities.tolist())
print("sum:", probabilities.sum().item(), "manual match:", torch.allclose(probabilities, manual))

probabilities: [0.6652409434318542, 0.2447284758090973, 0.09003057330846786]
sum: 1.0 manual match: True


In [4]:
from opd_study.models import TinyCausalLM, TinyTransformerConfig

config = TinyTransformerConfig(vocab_size=12, max_sequence_length=16,
                               number_of_layers=1, hidden_size=16,
                               number_of_heads=4, feed_forward_size=32)
model = TinyCausalLM(config).eval()
prefix_a = torch.tensor([[1, 4, 5, 6]])
prefix_b = prefix_a.clone(); prefix_b[0, -1] = 7
with torch.no_grad():
    logits_a, logits_b = model(prefix_a), model(prefix_b)
print("shape:", tuple(logits_a.shape),
      "past unchanged:", torch.allclose(logits_a[:, :-1], logits_b[:, :-1]))

shape: (1, 4, 12) past unchanged: True


## Checks

In [5]:
assert probabilities.shape == (3,)
assert torch.isclose(probabilities.sum(), torch.tensor(1.0))
assert torch.allclose(logits_a[:, :-1], logits_b[:, :-1])
print("check passed: normalized next-token probabilities and causal prefix")

check passed: normalized next-token probabilities and causal prefix


**Exercise (5 min):** change the first rather than last token of `prefix_b`. Predict which logits may change and write an assertion.

<details><summary>Check</summary>Positions after the changed token may differ; no future token may alter an earlier logit.</details>

## My recurring mistakes

### M1 — Treating logits as probabilities

- Wrong: assume logits already sum to one.
- Why: logits are unnormalized real scores.
- Fix: apply vocabulary-axis softmax and check the sum.
- Related check: `test_future_token_does_not_change_past_logits`

### M2 — Merging causal and response masks

- Wrong: assume blocking the future also removes prompt loss.
- Why: attention visibility and objective inclusion differ.
- Fix: audit causal, attention, and response masks separately.
- Related check: `test_sft_counts_only_response_targets`

## 60-second summary

1. turn logits into probabilities
2. read next-token tensor shapes
3. verify causal prefixes

## Next Steps

Before the next notebook, rerun the assertions and record one prediction you revised.

### Sources

- [`gkd`](https://arxiv.org/abs/2306.13649v3) · `2306.13649v3` · license `CC-BY-4.0` · [audited manifest](../../docs/sources.yml)